# Digit Recognizer — *casa Zaccaria* notebook

Goal is **learning**, not chasing the leaderboard: the CNN dojo before real OCR/HTR.
We grow the pipeline version by version, each step with a **pre-declared success criterion**
and a logged **ADOPT / REJECT** verdict.

**House rules applied here**
- The out-of-fold (OOF) score is the *authoritative* signal. The public leaderboard is noise.
- Compare **like-for-like**: single-model OOF vs single-model OOF; **same training regime on both sides**.
- One variable per experiment, so we always know *which* change moved the needle.
- Pre-declared criterion written *before* results; a FAIL is logged honestly, never re-rolled into a PASS.
- **Consolidation rule:** an effect is real only if it **replicates**. Single-split screening *ranks*
  candidates, it never *concludes*. (Section 8 documents what happened when we forgot this.)
- The experiment log lives inside this notebook; it is the write-up skeleton.

**Headline result:** v0 0.9207 → v3 OOF 0.9909 (LB 0.99460). Where the gains actually came from is
decomposed in Section 11 — and it is *not* where we first assumed.

## 1. Setup

Reproducible seeds and a single config block. Set the Kaggle accelerator to **GPU**
(Settings → Accelerator) before the CNN sections.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5
DATA_DIR = "/kaggle/input/competitions/digit-recognizer"   # user-specified mount path

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("Setup done. Seed:", SEED, "| DATA_DIR:", DATA_DIR)

## 2. Load data

785 columns: one `label` + 784 flattened pixels (28×28). Pixels 0–255 → scale to [0,1].
Scaling is not "cleaning"; it just helps the optimizer converge.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

y = train["label"].values
X = train.drop(columns="label").values.astype("float32") / 255.0   # (n, 784)
X_test = test.values.astype("float32") / 255.0                     # (m, 784)
print("train:", X.shape, " test:", X_test.shape, " labels:", np.unique(y))

## 3. Quick EDA — *and the validation decision*

Two questions, both of which drive the validation scheme:

1. **Balanced classes?** If yes, plain accuracy is honest and no imbalance machinery is needed.
2. **Independent samples?** In *Trace the Ace* samples shared a `session_id`, forcing `GroupKFold`.
   Here each image is an independent digit — no grouping key — so the correct choice is
   `StratifiedKFold` (stratified on the label to keep each fold's class mix representative).

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(y).value_counts().sort_index()
print(counts.to_dict())
print("min/max class share:", round(counts.min()/len(y), 4), "/", round(counts.max()/len(y), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(counts.index, counts.values)
ax[0].set_title("Class balance"); ax[0].set_xlabel("digit"); ax[0].set_ylabel("count")
ax[0].set_xticks(range(10))

grid = np.zeros((28, 28*10))
for d in range(10):
    grid[:, d*28:(d+1)*28] = X[y == d][0].reshape(28, 28)
ax[1].imshow(grid, cmap="gray_r"); ax[1].set_title("One sample per class"); ax[1].axis("off")
plt.tight_layout(); plt.show()

**Decision (logged):** classes ~balanced (~10% each) → accuracy is fair, no correction.
Samples independent → `StratifiedKFold(n_splits=5, shuffle=True)`. This *same split object* is
reused across every version so OOF scores are directly comparable version-to-version.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def write_submission(pred, filename):
    sub = pd.DataFrame({"ImageId": np.arange(1, len(pred) + 1), "Label": pred})
    sub.to_csv(filename, index=False)
    print(f"{filename} written:", len(sub), "rows")
    return sub

## 4. v0 — logistic regression baseline

The floor. A linear model on raw pixels treats pixel (5,5) and (5,6) as unrelated columns —
no notion of *where* things are. Whatever it reaches is the ceiling of the
"an image is 784 scalars" mindset.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

oof_v0 = cross_val_predict(
    LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1),
    X, y, cv=skf, method="predict", n_jobs=-1,
)
cv_v0 = accuracy_score(y, oof_v0)
print("v0 OOF accuracy:", round(cv_v0, 4))   # recorded: 0.9207  (LB 0.9205 -> pipeline clean)

clf = LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1).fit(X, y)
_ = write_submission(clf.predict(X_test), "submission_v0.csv")

## 5. The CNN

Reshape the 784 columns into a 28×28×1 **image** and let convolutions learn features via
**locality** (small 3×3 filters look at one neighborhood) and **weight sharing** (the same filter
slides everywhere → translation tolerance, reinforced by `MaxPooling`).

`build_cnn` is defined **once**, with augmentation knobs and a `deep` flag, all defaulting to off.
One definition serves every version, so architecture is provably identical across experiments unless
a flag says otherwise — and each experiment flips exactly one flag. Augmentation runs in training
only (like Dropout); at inference it is a no-op, so every OOF/test prediction is on **clean images**.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)

X_img = X.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)


def build_cnn(translate=0.0, rotate=0.0, zoom=0.0, deep=False, dropout=0.0):
    # Augmentation covers POSE (position, orientation, scale), not IDENTITY.
    # All defaults off => the bare v1 architecture.
    aug = []
    if translate: aug.append(layers.RandomTranslation(translate, translate))
    if rotate:    aug.append(layers.RandomRotation(rotate))
    if zoom:      aug.append(layers.RandomZoom(zoom))

    inputs = keras.Input(shape=(28, 28, 1))
    x = keras.Sequential(aug, name="aug")(inputs) if aug else inputs

    for filters in (32, 64):
        x = layers.Conv2D(filters, 3, padding="same" if deep else "valid", use_bias=not deep)(x)
        if deep: x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        if deep:                                   # second conv per block only in `deep`
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)               # keep strongest response per region
        if dropout: x = layers.Dropout(dropout)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128)(x)
    if deep: x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    if dropout: x = layers.Dropout(dropout * 2)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    m = keras.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


build_cnn().summary()

### v1 — first CNN (no augmentation, EarlyStopping on `val_accuracy`)

Deliberately minimal so the v0 → v1 delta cleanly isolates *"the model now knows it is looking
at an image."*

**Pre-declared criterion:** beat v0's OOF (0.9207).  *Result: 0.9735 → PASS (+0.0528).*

In [ ]:
oof_v1 = np.zeros(len(y), dtype="int64")
test_probs_v1 = np.zeros((len(X_test_img), 10), dtype="float32")
early_acc = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                          restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn()                                  # bare CNN
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=20, batch_size=128, callbacks=[early_acc], verbose=2)
    oof_v1[va] = model.predict(X_img[va], verbose=0).argmax(1)      # single-model OOF (authoritative)
    test_probs_v1 += model.predict(X_test_img, verbose=0) / N_SPLITS  # 5-model bag (submission only)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1[va]), 4))

cv_v1 = accuracy_score(y, oof_v1)
print("\nv1 OOF accuracy:", round(cv_v1, 4), "| delta vs v0:", round(cv_v1 - cv_v0, 4),
      "|", "PASS" if cv_v1 > cv_v0 else "FAIL")
_ = write_submission(test_probs_v1.argmax(1), "submission_v1.csv")

> **OOF vs LB.** The OOF (0.9735) is a *single model per fold*; the submission is a *5-model bag*.
> The LB (0.98025) sitting above the OOF is the bagging lift, not overfitting. Decisions use
> single-model OOF, compared like-for-like.

### v1 error analysis — measure, don't eyeball

A confusion matrix is **directional** (`i → j` ≠ `j → i`). Pair counts vary run-to-run, so compare
*within* a run — and, per the consolidation rule, never conclude from a single one.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_v1 = confusion_matrix(y, oof_v1)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ConfusionMatrixDisplay(cm_v1, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("v1 OOF confusion matrix"); plt.tight_layout(); plt.show()

off = cm_v1.copy(); np.fill_diagonal(off, 0)

pairs = sorted(((i, j, off[i, j]) for i in range(10) for j in range(10) if off[i, j] > 0),
               key=lambda t: t[2], reverse=True)
print("Top directional confusions (true -> pred):")
for i, j, n in pairs[:6]:
    print(f"  {i} -> {j}: {n:>3}  ({n / cm_v1[i].sum() * 100:.2f}% of the {i}s)")

sym = {(i, j): off[i, j] + off[j, i] for i in range(10) for j in range(i + 1, 10)}
print("\nTop confusable pairs (both directions):")
for (i, j), n in sorted(sym.items(), key=lambda t: t[1], reverse=True)[:5]:
    print(f"  {i} <-> {j}: {n:>3}  ({off[i, j]} one way, {off[j, i]} the other)")

## 6. v1b — controlled ablation: EarlyStopping on `val_loss`

**One variable changed vs v1:** monitor `val_loss` (smooth) instead of `val_accuracy` (a staircase
metric that plateaus and can stop training too early).

**Pre-declared criterion:** beat v1's OOF (0.9735).  *Result: 0.9811 → PASS (+0.0059).* Note a single
model at 0.9811 beats v1's 5-model bag (LB 0.98025): **train better before bagging more.**

In [ ]:
oof_v1b = np.zeros(len(y), dtype="int64")
early_loss = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                           restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn()
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=25, batch_size=128, callbacks=[early_loss], verbose=2)
    oof_v1b[va] = model.predict(X_img[va], verbose=0).argmax(1)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1b[va]), 4))

cv_v1b = accuracy_score(y, oof_v1b)
print("\nv1b OOF accuracy:", round(cv_v1b, 4), "| delta vs v1:", round(cv_v1b - cv_v1, 4),
      "|", "PASS" if cv_v1b > cv_v1 else "FAIL")

## 7. v2 — naive data augmentation (all three knobs at once)

Add pose variation (shift + rotate + zoom together) on the v1b base.

**Pre-declared criterion:** beat v1b (0.9811).
**Sub-hypothesis (before results):** augmentation reduces 4↔9 and 8↔9, leaves 2↔7 ~unchanged.

> **Result: 0.9704 → REJECT (−0.0107).** Kept as-run; the FAIL is diagnosed, not re-rolled.
> Fold epoch counts (19 / 5 / 5 / 6 / 6) exposed the cause: the augmented `val_loss` is noisy early,
> `patience=6` tripped, and `restore_best_weights` reverted 4 of 5 folds to nearly untrained
> checkpoints. The result is **confounded by early stopping** — the stop point is a hidden variable
> that *changes with augmentation*. Hence the fixed-epoch design from Section 8 on.

In [ ]:
oof_v2 = np.zeros(len(y), dtype="int64")
test_probs_v2 = np.zeros((len(X_test_img), 10), dtype="float32")
early_v2 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                         restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn(translate=0.10, rotate=0.04, zoom=0.10)   # all three knobs on
    hist = model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
                     epochs=40, batch_size=128, callbacks=[early_v2], verbose=2)
    oof_v2[va] = model.predict(X_img[va], verbose=0).argmax(1)
    test_probs_v2 += model.predict(X_test_img, verbose=0) / N_SPLITS
    print(f"fold {fold}: acc={round(accuracy_score(y[va], oof_v2[va]), 4)} "
          f"| epochs_run={len(hist.history['loss'])}")   # the smoking gun

cv_v2 = accuracy_score(y, oof_v2)
print("\nv2 OOF accuracy:", round(cv_v2, 4), "| delta vs v1b:", round(cv_v2 - cv_v1b, 4),
      "|", "PASS" if cv_v2 > cv_v1b else "FAIL")
_ = write_submission(test_probs_v2.argmax(1), "submission_v2.csv")

In [ ]:
cm_v2 = confusion_matrix(y, oof_v2)

def pair(cm, i, j):
    return cm[i, j] + cm[j, i]

print("Pre-declared pairs (both directions), v1 -> v2:")
for i, j in [(2, 7), (7, 9), (4, 9), (8, 9)]:
    b, a = pair(cm_v1, i, j), pair(cm_v2, i, j)
    print(f"  {i}<->{j}: {b} -> {a}  ({a - b:+d})")

diff = (cm_v2 - cm_v1).astype(int); np.fill_diagonal(diff, 0)
m = max(1, int(np.abs(diff).max()))
fig, ax = plt.subplots(figsize=(5.4, 5.4))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-m, vmax=m)
for i in range(10):
    for j in range(10):
        if diff[i, j]:
            ax.text(j, i, f"{diff[i, j]:+d}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Off-diagonal change: v2 - v1  (blue = fewer errors)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

*Observed on our run:* 7↔9 **+49**, 4↔9 −28, 2↔7 +7, 8↔9 +0. The 7↔9 spike was initially blamed on
rotation. Section 8 shows that attribution was **wrong** — it was the early-stopping confound.

## 8. Augmentation ablations — one knob at a time, **run twice**

The naive v2 mixed three knobs *and* an early-stopping confound. Here each knob is isolated under a
**fixed training regime** (fixed epochs, no EarlyStopping → the stop point stops being a hidden
variable), on a single stratified split.

**SCREEN mode:** a screening signal to *rank* knobs. Not the authoritative OOF, and — as the two runs
below prove — **not a basis for conclusions on its own**.

**Pre-declared hypothesis (before results):** translation helps or is neutral; zoom is mild; rotation
is the villain, expected to inflate **7↔9**.

Set `RUNS = 2` (or more): the same screen, different TF seeds. If an effect does not survive a rerun,
it was never there.

In [ ]:
from sklearn.model_selection import train_test_split

SCREEN_EPOCHS = 25
RUNS = 2                                   # replication is the point of this cell

configs = {
    "ref_noaug":   dict(),
    "a_translate": dict(translate=0.10),
    "b_rotate":    dict(rotate=0.04),
    "c_zoom":      dict(zoom=0.10),
}

Xtr, Xva, ytr, yva = train_test_split(X_img, y, test_size=0.2, stratify=y, random_state=SEED)
pair = lambda cm, i, j: cm[i, j] + cm[j, i]

all_runs = []
for run in range(RUNS):
    res = {}
    for name, cfg in configs.items():
        tf.random.set_seed(SEED + 1000 * run)      # same init per run; differs across runs
        model = build_cnn(**cfg)
        model.fit(Xtr, ytr, epochs=SCREEN_EPOCHS, batch_size=128, verbose=0)
        pred = model.predict(Xva, verbose=0).argmax(1)
        cm = confusion_matrix(yva, pred)
        res[name] = dict(acc=accuracy_score(yva, pred),
                         p79=pair(cm, 7, 9), p49=pair(cm, 4, 9), p27=pair(cm, 2, 7))
    all_runs.append(res)

    ref = res["ref_noaug"]["acc"]
    print(f"\n=== run {run} ===")
    print(" config        val_acc   d_acc     7<->9  4<->9  2<->7")
    for name, r in res.items():
        print(f" {name:12s} {r['acc']:.4f}  {r['acc']-ref:+.4f}    {r['p79']:>4}   {r['p49']:>4}   {r['p27']:>4}")

# Consolidation check: an effect counts only if its SIGN is stable across runs.
print("\n=== consolidation: d_acc sign across runs ===")
for name in configs:
    if name == "ref_noaug":
        continue
    ds = [r[name]["acc"] - r["ref_noaug"]["acc"] for r in all_runs]
    stable = all(d > 0 for d in ds) or all(d < 0 for d in ds)
    print(f" {name:12s} " + "  ".join(f"{d:+.4f}" for d in ds) +
          f"   -> {'REPLICATED' if stable else 'NOT REPLICATED (noise)'}")

### Ablation verdict (recorded across two runs) — *and a correction*

| config | d_acc run 1 | d_acc run 2 | 4↔9 run 1 | 4↔9 run 2 | verdict |
|--------|:---:|:---:|:---:|:---:|---|
| ref_noaug | (0.9886) | (0.9902) | 4 | 3 | reference |
| a_translate | +0.0018 | +0.0004 | 5 | 6 | positive both runs, but tiny |
| b_rotate | **−0.0020** | **−0.0026** | **18** | **6** | **REPLICATED: rotation harms** |
| c_zoom | +0.0013 | −0.0002 | 5 | 5 | **sign flipped → not replicated** |

**What survives two runs: exactly one thing — rotation hurts.** Consistent in sign and magnitude.

**What does not survive — and the lesson:**
- The run-1 spike **4↔9 = 18** collapsed to **6** on rerun. It was **noise on a single split**, yet it
  had been used to build a confident geometric story ("rotation blurs an orientation-sensitive angular
  feature"). That story was **fabricated from noise**. Retracted.
- **Zoom flipped sign** (+0.0013 → −0.0002). The "zoom helps" verdict was noise too.
- Consequently the earlier claim that *"rotation is exonerated for v2's 7↔9 spike"* rests on pair
  counts too small to support it. The defensible statement is narrower: **v2's aggregate FAIL is
  explained by the early-stopping confound**, and per-pair attributions at this sample size are
  unreliable in either direction.

**Method note (the real takeaway).** Screening on one split ranks candidates; it does not license
conclusions. Effects must **replicate** before they enter the log as findings. Two runs caught a
false finding that was already written into a previous version of this notebook.

**Decision (justified by the one replicated effect):** drop rotation. Keep translation (positive in
both runs, though small). Zoom is retained in v3 only as part of the pre-registered combo, and its
individual contribution is explicitly **not** claimed.

## 9. v3 — translation + zoom, rotation dropped (full 5-fold, fixed-epoch regime)

Two things differ from v1b, so we run a **matched pair** on the same 5 folds and isolate each:

- **v1c** — no augmentation, **fixed-epoch regime**. Changing the regime is itself a variable, so this
  is the fair re-based reference. Without v1c, every gain would be misattributed to augmentation.
- **v3** — v1c **plus** translation + zoom. The only difference is augmentation.

**Pre-declared criteria:** v3 must beat (1) **v1c** (clean augmentation effect) and (2) the adopted
best **v1b = 0.9811**.

*Results: v1c **0.9897**, v3 **0.9909** → both PASS. LB 0.99460.*

In [ ]:
FINAL_EPOCHS = 30

final_configs = {
    "v1c_noaug_fixed":   dict(),                           # fair reference in this regime
    "v3_translate_zoom": dict(translate=0.10, zoom=0.10),  # one variable added: augmentation
}

final_oof, final_testprobs = {}, {}
for name, cfg in final_configs.items():
    oof = np.zeros(len(y), dtype="int64")
    tprobs = np.zeros((len(X_test_img), 10), dtype="float32")
    for fold, (tr, va) in enumerate(skf.split(X_img, y)):
        tf.random.set_seed(SEED + fold)
        model = build_cnn(**cfg)
        model.fit(X_img[tr], y[tr], epochs=FINAL_EPOCHS, batch_size=128, verbose=0)  # fixed, no ES
        oof[va] = model.predict(X_img[va], verbose=0).argmax(1)
        tprobs += model.predict(X_test_img, verbose=0) / N_SPLITS
    final_oof[name], final_testprobs[name] = oof, tprobs
    print(f"{name:18s} OOF: {accuracy_score(y, oof):.4f}")

cv_v1c = accuracy_score(y, final_oof["v1c_noaug_fixed"])
cv_v3 = accuracy_score(y, final_oof["v3_translate_zoom"])
print(f"\nv1c (fixed-epoch reference): {cv_v1c:.4f}")
print(f"v3  (translate + zoom):      {cv_v3:.4f}")
print(f"aug effect (v3 - v1c):       {cv_v3 - cv_v1c:+.4f}  |", "PASS" if cv_v3 > cv_v1c else "FAIL")
print(f"vs adopted best v1b:         {cv_v3 - cv_v1b:+.4f}  |", "PASS" if cv_v3 > cv_v1b else "FAIL")
_ = write_submission(final_testprobs["v3_translate_zoom"].argmax(1), "submission_v3.csv")

### Decomposing the +0.0098 (v1b → v3)

| source | delta | share |
|---|:---:|:---:|
| fixed-epoch regime (v1b → v1c) | **+0.0086** | ~89% |
| augmentation (v1c → v3) | +0.0011 | ~11% |

**Eight times more value came from not under-training than from augmentation.** This is precisely the
scenario the matched v1c/v3 pair was built to detect: without v1c, the entire jump would have been
credited to augmentation, and the actual lesson — *stop under-training* — would have been missed.

Caveat, stated plainly: **+0.0011 is ~46 images out of 42,000.** It passes both criteria, but it is a
small effect near the noise floor and has been measured once. Section 10 replicates it before it is
allowed to count as a finding.

## 10. Replication check — is the +0.0011 augmentation effect real?

Per the consolidation rule, an effect enters the log as a finding only if it **replicates**. We re-run
the matched v1c/v3 pair under different seeds and check that the *sign* of the augmentation effect is
stable. This is the discipline that Section 8 taught us the hard way.

**Pre-declared criterion:** the augmentation effect (v3 − v1c) must be **positive in every seed**.
Otherwise it is logged as **not consolidated** — regardless of how good the leaderboard looks.

In [ ]:
REPLICATION_SEEDS = [1337, 2024]        # add more if compute allows

effects = []
for s in REPLICATION_SEEDS:
    accs = {}
    for name, cfg in final_configs.items():
        oof = np.zeros(len(y), dtype="int64")
        for fold, (tr, va) in enumerate(skf.split(X_img, y)):
            tf.random.set_seed(s + fold)
            model = build_cnn(**cfg)
            model.fit(X_img[tr], y[tr], epochs=FINAL_EPOCHS, batch_size=128, verbose=0)
            oof[va] = model.predict(X_img[va], verbose=0).argmax(1)
        accs[name] = accuracy_score(y, oof)
    eff = accs["v3_translate_zoom"] - accs["v1c_noaug_fixed"]
    effects.append(eff)
    print(f"seed {s}: v1c={accs['v1c_noaug_fixed']:.4f}  v3={accs['v3_translate_zoom']:.4f}  "
          f"effect={eff:+.4f}")

effects_all = [cv_v3 - cv_v1c] + effects        # include the original run
print("\nall augmentation effects:", [f"{e:+.4f}" for e in effects_all])
print("verdict:", "CONSOLIDATED (positive in every seed)" if all(e > 0 for e in effects_all)
      else "NOT CONSOLIDATED (sign unstable -> treat as noise)")

## 11. v4 — BatchNorm + dropout + deeper head

The architectural lever, at last. Three changes ride together in the `deep` flag (a second conv per
block, BatchNorm everywhere, dropout), so v4 tests **"a better architecture"** as one package rather
than three separate knobs — an honest scope statement, not a claim about which sub-component matters.

Same lesson as before: v4 is compared against a **matched baseline in the same regime** (v3, fixed
epochs, same augmentation). Only the architecture changes.

BatchNorm and dropout regularise, so a deeper net tolerates — and needs — more epochs. That is a
coupled consequence of the architecture, not an independent knob.

**Pre-declared criteria (written before results):**
1. v4 must beat **v3 (0.9909)** on the same 5 folds.
2. The improvement must exceed **+0.0005** (~21 images) to count as more than noise; anything smaller
   is logged as **inconclusive**, not as a win.
3. If it passes, it must **replicate** across a second seed before adoption (Section 10's rule).

In [ ]:
V4_EPOCHS = 40      # BN + dropout regularise -> the deeper net trains longer

v4_cfg = dict(translate=0.10, zoom=0.10, deep=True, dropout=0.25)
build_cnn(**v4_cfg).summary()

In [ ]:
oof_v4 = np.zeros(len(y), dtype="int64")
test_probs_v4 = np.zeros((len(X_test_img), 10), dtype="float32")

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    tf.random.set_seed(SEED + fold)          # same seeds as v3 -> paired comparison
    model = build_cnn(**v4_cfg)
    model.fit(X_img[tr], y[tr], epochs=V4_EPOCHS, batch_size=128, verbose=0)
    oof_v4[va] = model.predict(X_img[va], verbose=0).argmax(1)
    test_probs_v4 += model.predict(X_test_img, verbose=0) / N_SPLITS
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v4[va]), 4))

cv_v4 = accuracy_score(y, oof_v4)
delta = cv_v4 - cv_v3
print(f"\nv4 OOF accuracy: {cv_v4:.4f} | delta vs v3: {delta:+.4f}")
print("criterion 1 (beat v3):        ", "PASS" if delta > 0 else "FAIL")
print("criterion 2 (delta > +0.0005):", "PASS" if delta > 0.0005 else "INCONCLUSIVE / FAIL")
print(f"  (delta in images: {delta * len(y):.0f} of {len(y)})")
_ = write_submission(test_probs_v4.argmax(1), "submission_v4.csv")

### Where do v3 and v4 disagree?

If two model families share the *same* error set, the remaining loss is likely irreducible (label
noise, genuinely ambiguous glyphs). If they fail on different images, an ensemble has something to
gain. This shared-error diagnostic is the plateau test.

In [ ]:
err_v3 = final_oof["v3_translate_zoom"] != y
err_v4 = oof_v4 != y
both = err_v3 & err_v4

print(f"v3 errors: {err_v3.sum()}  |  v4 errors: {err_v4.sum()}  |  shared: {both.sum()}")
print(f"shared share of v4 errors: {both.sum() / max(1, err_v4.sum()):.1%}")
print("-> high overlap = approaching an irreducible floor; low overlap = ensembling should help.")

cm_v4 = confusion_matrix(y, oof_v4)
print("\nKey pairs, v3 -> v4:")
cm_v3 = confusion_matrix(y, final_oof["v3_translate_zoom"])
for i, j in [(2, 7), (7, 9), (4, 9), (8, 9)]:
    b, a = pair(cm_v3, i, j), pair(cm_v4, i, j)
    print(f"  {i}<->{j}: {b} -> {a}  ({a - b:+d})")

# The images both models get wrong: eyeball whether a human could do better.
idx = np.where(both)[0][:12]
if len(idx):
    fig, axes = plt.subplots(2, 6, figsize=(11, 4))
    for ax_, k in zip(axes.ravel(), idx):
        ax_.imshow(X_img[k].squeeze(), cmap="gray_r")
        ax_.set_title(f"y={y[k]} p={oof_v4[k]}", fontsize=9)
        ax_.axis("off")
    plt.suptitle("Images both v3 and v4 get wrong")
    plt.tight_layout(); plt.show()

## 12. Experiment log  *(the write-up skeleton)*

Criterion written **before** results; verdict **after**. Rows marked *(screen)* are single-split
val_acc and are **not** comparable to 5-fold OOF.

| Version | Description | CV (OOF acc) | Pre-declared criterion | Verdict |
|--------:|-------------|:------------:|------------------------|:-------:|
| v0  | Logistic regression on raw pixels | 0.9207 | baseline floor | REFERENCE |
| v1  | Minimal CNN, ES on `val_accuracy` | 0.9735 | beat v0 (0.9207) | ADOPT (+0.0528) |
| v1b | v1 with ES on `val_loss` | 0.9811 | beat v1 (0.9735) | ADOPT (+0.0059) |
| v2  | v1b + naive augmentation (3 knobs) | 0.9704 | beat v1b (0.9811) | REJECT (−0.0107, confounded) |
| v2a | ablation: translation *(screen ×2)* | +0.0018 / +0.0004 | rank knobs | positive, small |
| v2b | ablation: rotation *(screen ×2)* | −0.0020 / −0.0026 | rank knobs | **REJECT — replicated harm** |
| v2c | ablation: zoom *(screen ×2)* | +0.0013 / −0.0002 | rank knobs | **NOT REPLICATED (noise)** |
| v1c | no-aug, fixed-epoch regime, 5-fold | 0.9897 | fair reference for regime | REFERENCE |
| v3  | v1c + translation + zoom, 5-fold | **0.9909** | beat v1c **and** v1b | **ADOPT — LB 0.99460** |
| v4  | v3 + BatchNorm/dropout/deeper (`deep`) | *fill* | beat v3 by > +0.0005, then replicate | *fill* |

**Running notes**
- v0: OOF 0.9207 ≈ LB 0.9205 → pipeline clean, OOF trustworthy from the start.
- v1: OOF 0.9735 (single) vs LB 0.98025 (5-model bag). Gap = bagging lift, not overfitting.
- v1b: a single model (0.9811) beats v1's 5-model bag → **train better before bagging more**.
- v2 REJECT. Fold epochs 19/5/5/6/6 → early stopping reverted 4/5 folds to under-trained checkpoints.
  The stop point was a hidden variable that changed with augmentation.
- **Section 8 correction.** Run 1 showed 4↔9 = 18 under rotation and a confident geometric explanation
  was written for it. Run 2 gave 6. It was noise; the explanation was fabricated. Zoom's sign flipped
  too. **Only "rotation harms" replicated.** Screening ranks; it does not conclude.
- v3 decomposition: **+0.0086 from the fixed-epoch regime, +0.0011 from augmentation.** The dominant
  lever across this whole notebook was *under-training*, twice disguised as something else (first as an
  early-stopping metric choice, then as "augmentation doesn't work").
- 2↔7 never moved in any experiment → the leading candidate for *irreducible* glyph ambiguity.

**Method lessons, portable beyond MNIST**
1. Compare like-for-like: same regime on both sides, or the credit lands on the wrong change.
2. Screening ≠ evidence. Replicate before believing, especially a number that flatters a nice story.
3. A pre-declared criterion is what makes a FAIL informative instead of embarrassing.
4. The most expensive bug in this notebook was never in the model — it was in the *training regime*.

## 13. Next steps

- Run Section 10; if the augmentation effect is **not** consolidated, downgrade the v3 log row to
  "adopted for the regime change; augmentation contribution not established". Be willing to do that
  even though the LB says 0.99460 — the LB does not adjudicate causes.
- If v4 passes both criteria, replicate on a second seed before adoption.
- Light **Optuna** tuning afterwards, always enqueuing the current best as an anti-regression trial.
- Then: **MuseumSCAT** (ECCV'26 CVNH). Not digit classification but sequence recognition (HTR) —
  variable-length output, CER/WER, handwriting. The methodology transfers; the architecture does not.